# 1. Introdução

Este notebook apresenta uma analise computacional de tres problemas de escalonamento: Machine Scheduling, Job Shop Scheduling e Flow Shop Scheduling.

Os modelos foram implementados em Julia com JuMP e resolvidos com o solver HiGHS. As instancias sao lidas a partir dos diretorios de dados do repositorio, enquanto os resultados da atividade sao armazenados exclusivamente em `desenvolvimento`.


## Identificação da atividade

- Disciplina: Otimizacao
- Atividade: Trabalho 02 - Analise computacional de problemas de escalonamento
- Problemas estudados:
  - Machine Scheduling
  - Job Shop Scheduling
  - Flow Shop Scheduling
- Linguagem: Julia
- Modelagem: JuMP
- Solver: HiGHS
- Pasta de desenvolvimento: `desenvolvimento`

# 2. Objetivos

Os objetivos deste trabalho sao:

1. Formular matematicamente tres problemas classicos de escalonamento.
2. Implementar os modelos em Julia usando JuMP.
3. Resolver instancias de referencia com o solver HiGHS.
4. Comparar resultados computacionais entre familias de instancias.
5. Discutir desempenho, qualidade das solucoes e limitacoes dos modelos.

# 3. Ambiente computacional

Esta secao registra informacoes obtidas por execucao no ambiente Julia usado no notebook. A versao da Julia e as bibliotecas carregadas sao consultadas diretamente pelas celulas abaixo.


In [ ]:
# Ambiente computacional

VERSION


## Bibliotecas utilizadas

As bibliotecas utilizadas nesta etapa sao:

- `JuMP`: modelagem do problema de otimizacao.
- `HiGHS`: solver MILP usado para resolver o modelo.
- `Printf`: formatacao de saidas numericas.
- `Statistics`: apoio a analises agregadas, quando necessario.

Os leitores das instancias permanecem implementados com Julia base para evitar dependencia de pacotes de leitura nesta etapa.


In [ ]:
# Bibliotecas utilizadas nesta etapa

using JuMP
using HiGHS
using Printf
using Statistics


# 4. Configuração do HiGHS

A configuracao do solver e mantida explicita para reproducibilidade. As funcoes de resolucao aceitam limite de tempo, tolerancia relativa de gap e opcao de silenciar o log do HiGHS.


In [ ]:
# Configuracao explicita do JuMP e do HiGHS

const SOLVER = HiGHS.Optimizer
const DEFAULT_TIME_LIMIT = 60.0

function configurar_solver!(model::Model; time_limit::Float64 = DEFAULT_TIME_LIMIT,
        gap_relativo = nothing, silent::Bool = true)
    if silent
        set_silent(model)
    end
    set_optimizer_attribute(model, "time_limit", time_limit)
    if gap_relativo !== nothing
        set_optimizer_attribute(model, "mip_rel_gap", float(gap_relativo))
    end
    return model
end


## Caminhos das instâncias

As instancias estao localizadas em `materiais/AULA06`. Todos os arquivos produzidos pela atividade devem permanecer dentro de `desenvolvimento`. As instancias serao apenas lidas, nunca modificadas.

In [ ]:
# Caminhos relativos a partir da raiz do repositorio

const ROOT_DIR = normpath(joinpath(@__DIR__, ".."))
const DATA_DIR = joinpath(ROOT_DIR, "materiais", "AULA06")

const MACHINE_DIR = joinpath(DATA_DIR, "machinescheduling_instances")
const JSP_DIR = joinpath(DATA_DIR, "jsplib_subset")
const FSSP_DIR = joinpath(DATA_DIR, "fssp_problems")

for dir in [MACHINE_DIR, JSP_DIR, FSSP_DIR]
    isdir(dir) || error("Diretorio de instancias nao encontrado: $dir")
end


# 5. Machine Scheduling


## Descrição do problema

As instancias de `machinescheduling_instances` representam a variante `1|r_j|sum T_j` de Machine Scheduling.

Ha uma unica maquina, um conjunto de tarefas independentes e, para cada tarefa `j`, sao dados:

- instante de liberacao `r_j`;
- tempo de processamento `p_j`;
- prazo `d_j`.

A maquina pode processar no maximo uma tarefa por vez. Uma tarefa so pode iniciar apos seu instante de liberacao. O objetivo principal da variante presente nos arquivos e minimizar a soma dos atrasos `T_j = max(0, C_j - d_j)`.

Embora o makespan nao seja a funcao objetivo desta variante, ele sera reportado como metrica complementar da solucao encontrada.


## 5.1 Formulação

Conjuntos:

- `J`: conjunto de tarefas.
- `P = {(i,j) in J x J : i < j}`: pares nao ordenados de tarefas usados nas disjuncoes.

Parametros:

- `r_j`: instante de liberacao da tarefa `j`.
- `p_j`: tempo de processamento da tarefa `j`.
- `d_j`: prazo da tarefa `j`.
- `H`: horizonte de tempo usado como limitante superior.
- `M`: constante big-M para ativar/desativar restricoes de precedencia entre pares.

Neste trabalho, usa-se `H = max_j r_j + sum_j p_j`, que e um limite superior simples para uma agenda sequencial viavel, e `M = H`.


### Variáveis de decisão

Para cada tarefa `j in J`:

- `s_j >= 0`: instante de inicio da tarefa.
- `C_j >= 0`: instante de conclusao da tarefa.
- `T_j >= 0`: atraso da tarefa.

Para cada par `(i,j) in P`:

- `x_ij in {0,1}`: variavel binaria de ordem. Se `x_ij = 1`, a tarefa `i` e processada antes da tarefa `j`; se `x_ij = 0`, a tarefa `j` e processada antes da tarefa `i`.


### Função objetivo

A funcao objetivo da variante `1|r_j|sum T_j` e minimizar a soma dos atrasos:

$$
\min \sum_{j \in J} T_j
$$


### Restrições

Liberacao das tarefas:

$$
s_j \ge r_j \quad \forall j \in J
$$

Definicao dos tempos de conclusao:

$$
C_j = s_j + p_j \quad \forall j \in J
$$

Definicao linear do atraso:

$$
T_j \ge C_j - d_j \quad \forall j \in J
$$

Como `T_j >= 0`, as duas restricoes representam `T_j = max(0, C_j - d_j)` no otimo.

Nao sobreposicao na maquina unica, para todo par `(i,j) in P`:

$$
C_i \le s_j + M(1 - x_{ij})
$$

$$
C_j \le s_i + Mx_{ij}
$$

Dominio das variaveis:

$$
s_j, C_j, T_j \ge 0, \quad x_{ij} \in \{0,1\}
$$


## Leitura das instâncias

As instancias de Machine Scheduling estao em arquivos JSON. Cada arquivo contem nome da instancia, numero de tarefas, lista de tarefas, tempos de liberacao, duracoes e prazos.

In [ ]:
# Leitura das instancias de Machine Scheduling

"""
    ler_instancia_machine(path::AbstractString)

Le uma instancia de Machine Scheduling em JSON no formato usado em
`machinescheduling_instances`.

Retorna um `NamedTuple` com os campos:

- `name`: nome da instancia;
- `n`: numero de tarefas;
- `jobs`: vetor com os nomes das tarefas;
- `release`: vetor com os instantes de liberacao;
- `duration`: vetor com os tempos de processamento;
- `due`: vetor com os prazos;
- `machines`: numero de maquinas, igual a 1;
- `operations`: numero de operacoes, igual ao numero de tarefas.

A funcao valida arquivo inexistente, arquivo vazio, campos obrigatorios ausentes,
vetores com tamanhos inconsistentes e valores numericos invalidos.
"""
function ler_instancia_machine(path::AbstractString)
    validar_arquivo_legivel(path)
    text = read(path, String)

    name = extrair_json_string(text, "name")
    n = extrair_json_int(text, "n")
    jobs = extrair_json_array_strings(text, "jobs")
    release = extrair_json_array_ints(text, "release")
    duration = extrair_json_array_ints(text, "duration")
    due = extrair_json_array_ints(text, "due")

    if n <= 0
        error("Instancia Machine Scheduling invalida: n deve ser positivo em $path")
    end
    if length(jobs) != n || length(release) != n || length(duration) != n || length(due) != n
        error("Instancia Machine Scheduling incompleta: tamanhos de jobs/release/duration/due nao coincidem com n em $path")
    end
    if any(x -> x < 0, release) || any(x -> x <= 0, duration) || any(x -> x < 0, due)
        error("Instancia Machine Scheduling invalida: tempos devem ser nao negativos e duracoes positivas em $path")
    end

    return (
        name = name,
        n = n,
        jobs = jobs,
        release = release,
        duration = duration,
        due = due,
        machines = 1,
        operations = n,
        path = path,
    )
end

function validar_arquivo_legivel(path::AbstractString)
    if !isfile(path)
        error("Arquivo nao encontrado: $path")
    end
    if filesize(path) == 0
        error("Arquivo vazio: $path")
    end
    return nothing
end

function extrair_json_string(text::AbstractString, key::AbstractString)
    m = match(Regex("\\\"" * key * "\\\"\\s*:\\s*\\\"([^\\\"]*)\\\""), text)
    isnothing(m) && error("Campo obrigatorio ausente ou invalido no JSON: $key")
    return String(m.captures[1])
end

function extrair_json_int(text::AbstractString, key::AbstractString)
    m = match(Regex("\\\"" * key * "\\\"\\s*:\\s*(-?\\d+)"), text)
    isnothing(m) && error("Campo obrigatorio ausente ou invalido no JSON: $key")
    return parse(Int, m.captures[1])
end

function extrair_json_array_raw(text::AbstractString, key::AbstractString)
    m = match(Regex("\\\"" * key * "\\\"\\s*:\\s*\\[([^\\]]*)\\]", "s"), text)
    isnothing(m) && error("Campo obrigatorio ausente ou invalido no JSON: $key")
    return m.captures[1]
end

function extrair_json_array_strings(text::AbstractString, key::AbstractString)
    raw = extrair_json_array_raw(text, key)
    vals = [String(m.captures[1]) for m in eachmatch(r"\"([^\"]*)\"", raw)]
    isempty(vals) && error("Array JSON vazio ou invalido: $key")
    return vals
end

function extrair_json_array_ints(text::AbstractString, key::AbstractString)
    raw = extrair_json_array_raw(text, key)
    tokens = [strip(t) for t in split(raw, ",") if !isempty(strip(t))]
    isempty(tokens) && error("Array JSON vazio ou invalido: $key")
    vals = Int[]
    for token in tokens
        if isnothing(match(r"^-?\d+$", token))
            error("Valor nao inteiro no array $key: $token")
        end
        push!(vals, parse(Int, token))
    end
    return vals
end


## 5.2 Implementação

A implementacao abaixo constroi o modelo MILP de Machine Scheduling, resolve a instancia com HiGHS e retorna a agenda, o valor objetivo, o makespan, o gap, o tempo de solucao e informacoes auxiliares do modelo.


In [ ]:
# Modelo de Machine Scheduling: 1|r_j|sum T_j

"""
    resolver_machine_scheduling(instancia; time_limit = DEFAULT_TIME_LIMIT, silent = true)

Constroi e resolve o modelo MILP para a variante `1|r_j|sum T_j`.

Entrada:

- `instancia`: estrutura retornada por `ler_instancia_machine`.
- `time_limit`: limite de tempo do HiGHS, em segundos.
- `silent`: controla a saida do solver.

Retorna um `NamedTuple` com modelo JuMP, status, objetivo, makespan, gap,
tempo de execucao, agenda ordenada por inicio e dados auxiliares.
"""
function resolver_machine_scheduling(instancia; time_limit::Float64 = DEFAULT_TIME_LIMIT, gap_relativo = nothing, silent::Bool = true)
    n = instancia.n
    J = collect(1:n)
    pares = [(i, j) for i in J for j in J if i < j]

    r = Dict(j => instancia.release[j] for j in J)
    p = Dict(j => instancia.duration[j] for j in J)
    d = Dict(j => instancia.due[j] for j in J)

    H = maximum(instancia.release) + sum(instancia.duration)
    M = H

    model = Model(SOLVER)
    configurar_solver!(model; time_limit = time_limit, gap_relativo = gap_relativo, silent = silent)

    @variable(model, 0 <= s[J] <= H)
    @variable(model, 0 <= C[J] <= H)
    @variable(model, 0 <= T[J] <= H)
    @variable(model, x[pares], Bin)

    @constraint(model, [j in J], s[j] >= r[j])
    @constraint(model, [j in J], C[j] == s[j] + p[j])
    @constraint(model, [j in J], T[j] >= C[j] - d[j])

    @constraint(model, [(i, j) in pares], C[i] <= s[j] + M * (1 - x[(i, j)]))
    @constraint(model, [(i, j) in pares], C[j] <= s[i] + M * x[(i, j)])

    @objective(model, Min, sum(T[j] for j in J))

    tempo = @elapsed optimize!(model)

    status = termination_status(model)
    primal = primal_status(model)
    tem_solucao = has_values(model)
    objetivo = tem_solucao ? objective_value(model) : NaN
    gap = try
        relative_gap(model)
    catch
        NaN
    end

    agenda = NamedTuple[]
    if tem_solucao
        for j in J
            inicio = value(s[j])
            fim = value(C[j])
            atraso = value(T[j])
            push!(agenda, (
                job = instancia.jobs[j],
                machine = "M1",
                start = inicio,
                finish = fim,
                duration = p[j],
                release = r[j],
                due = d[j],
                tardiness = atraso,
            ))
        end
        sort!(agenda, by = a -> (a.start, a.finish, a.job))
    end

    makespan = tem_solucao ? maximum(a.finish for a in agenda) : NaN

    return (
        model = model,
        instance = instancia.name,
        status = status,
        primal_status = primal,
        objective = objetivo,
        makespan = makespan,
        solve_time = tempo,
        gap = gap,
        schedule = agenda,
        horizon = H,
        big_m = M,
        binary_variables = length(pares),
    )
end

"""
    validar_solucao_machine(instancia, solucao; atol = 1e-5)

Valida uma solucao de Machine Scheduling verificando liberacao, duracao,
atraso, nao sobreposicao e consistencia do makespan.

Retorna um `NamedTuple` com `valid`, lista de `violations` e `makespan_calculado`.
"""
function validar_solucao_machine(instancia, solucao; atol::Float64 = 1e-5)
    violacoes = String[]
    agenda = solucao.schedule

    if length(agenda) != instancia.n
        push!(violacoes, "numero de tarefas na agenda difere de n")
    end

    por_job = Dict(a.job => a for a in agenda)
    for j in 1:instancia.n
        job = instancia.jobs[j]
        if !haskey(por_job, job)
            push!(violacoes, "tarefa ausente na agenda: $job")
            continue
        end
        a = por_job[job]
        if a.machine != "M1"
            push!(violacoes, "tarefa $job alocada em maquina diferente de M1")
        end
        if a.start + atol < instancia.release[j]
            push!(violacoes, "tarefa $job inicia antes do release")
        end
        if abs((a.finish - a.start) - instancia.duration[j]) > atol
            push!(violacoes, "duracao inconsistente para tarefa $job")
        end
        atraso_calculado = max(0.0, a.finish - instancia.due[j])
        if abs(a.tardiness - atraso_calculado) > atol
            push!(violacoes, "atraso inconsistente para tarefa $job")
        end
    end

    ordenada = sort(agenda, by = a -> (a.start, a.finish, a.job))
    for k in 1:(length(ordenada) - 1)
        atual = ordenada[k]
        proxima = ordenada[k + 1]
        if atual.finish > proxima.start + atol
            push!(violacoes, "sobreposicao entre $(atual.job) e $(proxima.job)")
        end
    end

    makespan_calculado = isempty(agenda) ? NaN : maximum(a.finish for a in agenda)
    if !isnan(solucao.makespan) && abs(solucao.makespan - makespan_calculado) > atol
        push!(violacoes, "makespan reportado difere do makespan calculado")
    end

    return (
        valid = isempty(violacoes),
        violations = violacoes,
        makespan_calculado = makespan_calculado,
    )
end

function imprimir_resumo_machine(solucao, validacao)
    println("Machine Scheduling - ", solucao.instance)
    println("  status: ", solucao.status)
    println("  primal_status: ", solucao.primal_status)
    println("  objetivo soma_atrasos: ", round(solucao.objective; digits = 6))
    println("  makespan: ", round(solucao.makespan; digits = 6))
    println("  tempo_execucao_s: ", round(solucao.solve_time; digits = 6))
    println("  gap: ", isnan(solucao.gap) ? "NA" : string(round(solucao.gap; digits = 6)))
    println("  binarios: ", solucao.binary_variables)
    println("  validacao: ", validacao.valid)
    if !validacao.valid
        println("  violacoes: ", validacao.violations)
    end
    println("  alocacao:")
    for a in solucao.schedule
        @printf("    %-4s %-2s inicio=%8.3f fim=%8.3f dur=%3d release=%3d due=%3d atraso=%8.3f\n",
             a.job, a.machine, a.start, a.finish, a.duration, a.release, a.due, a.tardiness)
    end
end


## 5.3 Exemplo de solução

Nesta etapa, o modelo de Machine Scheduling sera executado inicialmente apenas para uma instancia pequena: `inst_n05_s01.json`. Os valores abaixo devem ser gerados pela execucao real do codigo, sem preenchimento manual.


In [ ]:
# Execucao inicial do Machine Scheduling em uma instancia pequena

inst_machine_pequena = ler_instancia_machine(joinpath(MACHINE_DIR, "inst_n05_s01.json"))
sol_machine_pequena = resolver_machine_scheduling(inst_machine_pequena; time_limit = 60.0, silent = true)
validacao_machine_pequena = validar_solucao_machine(inst_machine_pequena, sol_machine_pequena)

imprimir_resumo_machine(sol_machine_pequena, validacao_machine_pequena)


# 6. Job Shop Scheduling


## Descrição do problema

No Job Shop Scheduling, cada job e composto por uma sequencia propria de operacoes. Cada operacao deve ser processada em uma maquina especifica por um determinado tempo, respeitando a ordem tecnologica do job. Cada maquina pode processar no maximo uma operacao por vez.

O objetivo previsto e minimizar o makespan, isto e, o instante de conclusao da ultima operacao.

## 6.1 Formulação

A formulacao prevista usa:

- variaveis de inicio para cada operacao;
- restricoes de precedencia dentro de cada job;
- restricoes disjuntivas para pares de operacoes que usam a mesma maquina;
- variavel de makespan `Cmax`;
- objetivo de minimizar `Cmax`.

## Leitura das instâncias

As instancias seguem o formato JSPLIB. A primeira linha util contem o numero de jobs e maquinas. Cada linha seguinte descreve um job por pares `(maquina, tempo)`. Nos arquivos, as maquinas estao indexadas a partir de zero.

In [ ]:
# Leitura das instancias JSPLIB

"""
    ler_instancia_jsp(path::AbstractString)

Le uma instancia de Job Shop Scheduling no formato JSPLIB usado em
`jsplib_subset/instances`.

Linhas vazias e comentarios iniciados por `#` sao ignorados. A primeira linha util
deve conter `n m`. Cada uma das `n` linhas seguintes deve conter `2m` inteiros,
organizados como pares `(maquina, tempo)`.

Retorna um `NamedTuple` com `name`, `n`, `machines`, `jobs`, `operations` e `path`.
O campo `jobs` e um vetor em que cada posicao contem a sequencia de operacoes do
job como `NamedTuple`s `(machine, duration)`. As maquinas sao mantidas com a
indexacao original do arquivo, isto e, iniciando em zero.
"""
function ler_instancia_jsp(path::AbstractString)
    validar_arquivo_legivel(path)
    linhas = String[]
    for linha in eachline(path)
        s = strip(linha)
        if !isempty(s) && !startswith(s, "#")
            push!(linhas, s)
        end
    end

    isempty(linhas) && error("Instancia JSPLIB sem linhas de dados: $path")

    cabecalho = split(linhas[1])
    length(cabecalho) == 2 || error("Cabecalho JSPLIB invalido em $path: esperado 'n m'")
    n = parse(Int, cabecalho[1])
    m = parse(Int, cabecalho[2])
    n > 0 && m > 0 || error("Cabecalho JSPLIB invalido: n e m devem ser positivos em $path")

    if length(linhas) < n + 1
        error("Instancia JSPLIB incompleta: esperado $n jobs, encontrados $(length(linhas) - 1) em $path")
    end

    jobs = Vector{Vector{NamedTuple{(:machine, :duration), Tuple{Int, Int}}}}()
    for j in 1:n
        tokens = split(linhas[j + 1])
        length(tokens) == 2m || error("Job $j invalido em $path: esperado $(2m) inteiros, encontrados $(length(tokens))")
        nums = parse.(Int, tokens)
        ops = NamedTuple{(:machine, :duration), Tuple{Int, Int}}[]
        for k in 1:2:length(nums)
            machine = nums[k]
            duration = nums[k + 1]
            0 <= machine < m || error("Maquina invalida no job $j em $path: $machine fora de 0:$(m - 1)")
            duration > 0 || error("Duracao invalida no job $j em $path: $duration")
            push!(ops, (machine = machine, duration = duration))
        end
        push!(jobs, ops)
    end

    return (
        name = basename(path),
        n = n,
        machines = m,
        jobs = jobs,
        operations = n * m,
        path = path,
    )
end


## 6.2 Implementação

A implementacao abaixo constroi o modelo disjuntivo de Job Shop Scheduling. As operacoes de um mesmo job respeitam a ordem tecnologica, e pares de operacoes que disputam a mesma maquina sao separados por variaveis binarias e restricoes Big-M.


In [ ]:
# Modelo de Job Shop Scheduling: formulacao disjuntiva com minimizacao do makespan

"""
    resolver_job_shop(instancia; time_limit = DEFAULT_TIME_LIMIT, gap_relativo = nothing, silent = true)

Constroi e resolve o modelo MILP para Job Shop Scheduling.

Cada operacao `(j,k)` representa a operacao `k` do job `j`, com maquina e duracao
informadas pela instancia JSPLIB. Operacoes do mesmo job respeitam a ordem
tecnologica e operacoes que usam a mesma maquina sao separadas por restricoes
disjuntivas com Big-M.
"""
function resolver_job_shop(instancia; time_limit::Float64 = DEFAULT_TIME_LIMIT,
        gap_relativo = nothing, silent::Bool = true)
    n = instancia.n
    J = collect(1:n)
    OPS = [(j, k) for j in J for k in 1:length(instancia.jobs[j])]

    maquina = Dict((j, k) => instancia.jobs[j][k].machine for (j, k) in OPS)
    duracao = Dict((j, k) => instancia.jobs[j][k].duration for (j, k) in OPS)
    H = sum(values(duracao))

    pares_mesma_maquina = Tuple{Tuple{Int, Int}, Tuple{Int, Int}}[]
    for a in 1:length(OPS)-1
        for b in a+1:length(OPS)
            op_a = OPS[a]
            op_b = OPS[b]
            if maquina[op_a] == maquina[op_b]
                push!(pares_mesma_maquina, (op_a, op_b))
            end
        end
    end

    model = Model(SOLVER)
    configurar_solver!(model; time_limit = time_limit, gap_relativo = gap_relativo, silent = silent)

    @variable(model, 0 <= s[OPS] <= H)
    @variable(model, 0 <= Cmax <= H)
    @variable(model, y[1:length(pares_mesma_maquina)], Bin)

    for j in J
        for k in 1:(length(instancia.jobs[j]) - 1)
            @constraint(model, s[(j, k + 1)] >= s[(j, k)] + duracao[(j, k)])
        end
        ultimo = length(instancia.jobs[j])
        @constraint(model, Cmax >= s[(j, ultimo)] + duracao[(j, ultimo)])
    end

    for (idx, (op_a, op_b)) in enumerate(pares_mesma_maquina)
        @constraint(model, s[op_a] + duracao[op_a] <= s[op_b] + H * (1 - y[idx]))
        @constraint(model, s[op_b] + duracao[op_b] <= s[op_a] + H * y[idx])
    end

    @objective(model, Min, Cmax)

    tempo = @elapsed optimize!(model)

    status = termination_status(model)
    primal = primal_status(model)
    tem_solucao = has_values(model)
    makespan = tem_solucao ? value(Cmax) : NaN
    gap = try
        relative_gap(model)
    catch
        NaN
    end

    operacoes = NamedTuple[]
    if tem_solucao
        for (j, k) in OPS
            inicio = value(s[(j, k)])
            dur = duracao[(j, k)]
            push!(operacoes, (
                job = "J$j",
                operation = k,
                machine = "M$(maquina[(j, k)])",
                machine_index = maquina[(j, k)],
                start = inicio,
                finish = inicio + dur,
                duration = dur,
            ))
        end
        sort!(operacoes, by = op -> (op.start, op.machine_index, op.job, op.operation))
    end

    return (
        model = model,
        instance = instancia.name,
        status = status,
        primal_status = primal,
        makespan = makespan,
        solve_time = tempo,
        gap = gap,
        operations = operacoes,
        horizon = H,
        big_m = H,
        binary_variables = length(pares_mesma_maquina),
    )
end

"""
    validar_solucao_job_shop(instancia, solucao; atol = 1e-5)

Valida precedencias tecnologicas, conflitos de maquina e consistencia do makespan.
"""
function validar_solucao_job_shop(instancia, solucao; atol::Float64 = 1e-5)
    violacoes = String[]
    ops = solucao.operations
    if length(ops) != instancia.operations
        push!(violacoes, "numero de operacoes difere de n*m")
    end

    por_job_operacao = Dict((parse(Int, op.job[2:end]), op.operation) => op for op in ops)
    for j in 1:instancia.n
        for k in 1:length(instancia.jobs[j])
            chave = (j, k)
            if !haskey(por_job_operacao, chave)
                push!(violacoes, "operacao ausente: job $j operacao $k")
                continue
            end
            op = por_job_operacao[chave]
            esperado = instancia.jobs[j][k]
            if op.machine_index != esperado.machine
                push!(violacoes, "maquina inconsistente no job $j operacao $k")
            end
            if abs((op.finish - op.start) - esperado.duration) > atol
                push!(violacoes, "duracao inconsistente no job $j operacao $k")
            end
            if op.start < -atol
                push!(violacoes, "inicio negativo no job $j operacao $k")
            end
        end
        for k in 1:(length(instancia.jobs[j]) - 1)
            atual = get(por_job_operacao, (j, k), nothing)
            proxima = get(por_job_operacao, (j, k + 1), nothing)
            if atual !== nothing && proxima !== nothing && proxima.start + atol < atual.finish
                push!(violacoes, "precedencia violada no job $j entre operacoes $k e $(k + 1)")
            end
        end
    end

    for maq in 0:(instancia.machines - 1)
        ops_maquina = sort([op for op in ops if op.machine_index == maq], by = op -> (op.start, op.finish))
        for idx in 1:(length(ops_maquina) - 1)
            atual = ops_maquina[idx]
            proxima = ops_maquina[idx + 1]
            if atual.finish > proxima.start + atol
                push!(violacoes, "sobreposicao na maquina M$maq entre $(atual.job).$(atual.operation) e $(proxima.job).$(proxima.operation)")
            end
        end
    end

    makespan_calculado = isempty(ops) ? NaN : maximum(op.finish for op in ops)
    if !isnan(solucao.makespan) && abs(solucao.makespan - makespan_calculado) > atol
        push!(violacoes, "makespan reportado difere do makespan calculado")
    end

    return (
        valid = isempty(violacoes),
        violations = violacoes,
        makespan_calculado = makespan_calculado,
    )
end

function imprimir_resumo_job_shop(solucao, validacao)
    println("Job Shop Scheduling - ", solucao.instance)
    println("  status: ", solucao.status)
    println("  primal_status: ", solucao.primal_status)
    println("  makespan: ", round(solucao.makespan; digits = 6))
    println("  tempo_execucao_s: ", round(solucao.solve_time; digits = 6))
    println("  gap: ", isnan(solucao.gap) ? "NA" : string(round(solucao.gap; digits = 6)))
    println("  binarios: ", solucao.binary_variables)
    println("  validacao: ", validacao.valid)
    if !validacao.valid
        println("  violacoes: ", validacao.violations)
    end
end


## 6.3 Exemplo de solução

A rotina de Job Shop e utilizada nos experimentos em lote. Para evitar impressao extensa da programacao completa no notebook final, os resultados agregados das instancias aparecem nos CSVs e nas secoes de resumo, graficos e discussao.


# 7. Flow Shop Scheduling


## Descrição do problema

As instancias de Flow Shop Scheduling em `fssp_problems` estao em arquivos CSV. A primeira coluna identifica os jobs e as demais colunas identificam as maquinas `M1`, `M2`, ..., `Mm`. Cada celula contem o tempo de processamento do job naquela maquina.

Neste formato, todas as tarefas seguem a mesma ordem de maquinas: primeiro a coluna `M1`, depois `M2`, e assim sucessivamente. Portanto, o problema tratado nesta secao e o *permutation flow shop*: a sequencia de jobs e unica e e usada em todas as maquinas.

O objetivo e minimizar o makespan, isto e, o instante de termino do ultimo job na ultima maquina.


## 7.1 Formulação

Conjuntos:

- `J = {1, ..., n}`: conjunto de jobs.
- `K = {1, ..., n}`: posicoes da sequencia comum.
- `I = {1, ..., m}`: conjunto de maquinas, na ordem fixa do fluxo.

Parametros:

- `p_{ji}`: tempo de processamento do job `j` na maquina `i`.

Variaveis de decisao:

- `x_{jk} in {0,1}`: vale 1 se o job `j` ocupa a posicao `k` da sequencia.
- `C_{ki} >= 0`: instante de conclusao do job que esta na posicao `k` na maquina `i`.
- `Cmax >= 0`: makespan.

Cada job deve ocupar exatamente uma posicao:

$$
\sum_{k \in K} x_{jk} = 1 \quad \forall j \in J
$$

Cada posicao deve receber exatamente um job:

$$
\sum_{j \in J} x_{jk} = 1 \quad \forall k \in K
$$

O tempo de processamento da posicao `k` na maquina `i` e dado por:

$$
P_{ki} = \sum_{j \in J} p_{ji} x_{jk}
$$

Precedencia entre maquinas para o mesmo job posicionado em `k`:

$$
C_{ki} \ge C_{k,i-1} + P_{ki} \quad \forall k \in K, \; i = 2, ..., m
$$

Nao sobreposicao na mesma maquina entre posicoes consecutivas da sequencia:

$$
C_{ki} \ge C_{k-1,i} + P_{ki} \quad \forall k = 2, ..., n, \; i \in I
$$

Inicializacao do primeiro job na primeira maquina:

$$
C_{11} \ge P_{11}
$$

Definicao do makespan:

$$
Cmax \ge C_{n,m}
$$

Funcao objetivo:

$$
\min Cmax
$$

As restricoes acima impedem sobreposicao porque, em cada maquina, a operacao da posicao `k` so pode terminar depois que a posicao anterior terminou nessa mesma maquina. A ordem fixa das maquinas e respeitada porque a operacao na maquina `i` so pode terminar depois da conclusao da mesma posicao na maquina `i-1`.


## Leitura das instâncias

As instancias de Flow Shop estao em CSV. As linhas representam jobs, as colunas representam maquinas e as celulas contem tempos de processamento. A ordem das colunas define a ordem comum de processamento das tarefas.

Exemplo de cabecalho:

```text
,M1,M2,M3
```

Nesse caso, cada job passa por `M1`, depois `M2`, depois `M3`.


In [ ]:
# Leitura das instancias de Flow Shop Scheduling

"""
    ler_instancia_fssp(path::AbstractString)

Le uma instancia de Flow Shop Scheduling em CSV no formato usado em
`fssp_problems`.

A primeira coluna contem os nomes dos jobs e as demais colunas representam as
maquinas. Cada celula numerica e o tempo de processamento do job na maquina.

Retorna um `NamedTuple` com `name`, `n`, `machines`, `jobs`, `machine_names`,
`processing_times`, `operations` e `path`. A matriz `processing_times` tem
dimensao `n x machines`.
"""
function ler_instancia_fssp(path::AbstractString)
    validar_arquivo_legivel(path)
    linhas = [strip(l) for l in eachline(path) if !isempty(strip(l))]
    isempty(linhas) && error("Instancia FSSP sem linhas de dados: $path")

    header = split(linhas[1], ",")
    length(header) >= 2 || error("Cabecalho FSSP invalido em $path: esperado coluna de jobs e ao menos uma maquina")
    machine_names = String.(strip.(header[2:end]))
    any(isempty, machine_names) && error("Cabecalho FSSP invalido: nome de maquina vazio em $path")

    jobs = String[]
    rows = Vector{Vector{Int}}()
    m = length(machine_names)

    for (line_number, linha) in enumerate(linhas[2:end])
        cols = split(linha, ",")
        length(cols) == m + 1 || error("Linha FSSP $(line_number + 1) invalida em $path: esperado $(m + 1) colunas, encontradas $(length(cols))")
        job = strip(cols[1])
        isempty(job) && error("Linha FSSP $(line_number + 1) invalida: nome de job vazio em $path")
        tempos = Int[]
        for token in cols[2:end]
            s = strip(token)
            isnothing(match(r"^\d+$", s)) && error("Tempo FSSP invalido na linha $(line_number + 1) em $path: $s")
            valor = parse(Int, s)
            valor > 0 || error("Tempo FSSP deve ser positivo na linha $(line_number + 1) em $path")
            push!(tempos, valor)
        end
        push!(jobs, job)
        push!(rows, tempos)
    end

    n = length(jobs)
    n > 0 || error("Instancia FSSP sem jobs: $path")
    P = Matrix{Int}(undef, n, m)
    for j in 1:n, i in 1:m
        P[j, i] = rows[j][i]
    end

    return (
        name = splitext(basename(path))[1],
        n = n,
        machines = m,
        jobs = jobs,
        machine_names = machine_names,
        processing_times = P,
        operations = n * m,
        path = path,
    )
end


## 7.2 Implementação

O modelo abaixo usa uma formulacao posicional para o permutation flow shop. A variavel binaria `x[j,k]` escolhe qual job fica em cada posicao da sequencia. As variaveis `C[k,i]` calculam os tempos de conclusao de cada posicao em cada maquina.

Como a sequencia e comum a todas as maquinas, a nao sobreposicao e controlada pela ordem das posicoes: na mesma maquina, a posicao `k` so pode ser processada depois da posicao `k-1`.


In [ ]:
# Modelo de Flow Shop Scheduling: permutation flow shop com minimizacao do makespan

"""
    resolver_flow_shop(instancia; time_limit = DEFAULT_TIME_LIMIT, silent = true)

Constroi e resolve o modelo MILP para o permutation flow shop scheduling.

Retorna modelo, status, makespan, tempo, gap, sequencia e tabela de operacoes.
"""
function resolver_flow_shop(instancia; time_limit::Float64 = DEFAULT_TIME_LIMIT, gap_relativo = nothing, silent::Bool = true)
    n = instancia.n
    m = instancia.machines
    J = collect(1:n)
    K = collect(1:n)
    I = collect(1:m)
    P = instancia.processing_times

    H = sum(P)

    model = Model(SOLVER)
    configurar_solver!(model; time_limit = time_limit, gap_relativo = gap_relativo, silent = silent)

    @variable(model, x[J, K], Bin)
    @variable(model, 0 <= C[K, I] <= H)
    @variable(model, 0 <= Cmax <= H)

    @constraint(model, [j in J], sum(x[j, k] for k in K) == 1)
    @constraint(model, [k in K], sum(x[j, k] for j in J) == 1)

    proc(k, i) = sum(P[j, i] * x[j, k] for j in J)

    @constraint(model, C[1, 1] >= proc(1, 1))
    @constraint(model, [k in K, i in I; k > 1], C[k, i] >= C[k - 1, i] + proc(k, i))
    @constraint(model, [k in K, i in I; i > 1], C[k, i] >= C[k, i - 1] + proc(k, i))
    @constraint(model, Cmax >= C[n, m])

    @objective(model, Min, Cmax)

    tempo = @elapsed optimize!(model)

    status = termination_status(model)
    primal = primal_status(model)
    tem_solucao = has_values(model)
    makespan = tem_solucao ? value(Cmax) : NaN
    gap = try
        relative_gap(model)
    catch
        NaN
    end

    sequencia_indices = Int[]
    operacoes = NamedTuple[]
    if tem_solucao
        for k in K
            valores = [value(x[j, k]) for j in J]
            j_escolhido = argmax(valores)
            push!(sequencia_indices, j_escolhido)
        end

        # Reconstrucao semi-ativa: remove folgas artificiais possiveis nas variaveis C.
        conclusao = zeros(Float64, n, m)
        for k in K
            job_idx = sequencia_indices[k]
            for i in I
                inicio_minimo = 0.0
                if k > 1
                    inicio_minimo = max(inicio_minimo, conclusao[k - 1, i])
                end
                if i > 1
                    inicio_minimo = max(inicio_minimo, conclusao[k, i - 1])
                end
                fim = inicio_minimo + P[job_idx, i]
                conclusao[k, i] = fim
                push!(operacoes, (
                    position = k,
                    job = instancia.jobs[job_idx],
                    machine = instancia.machine_names[i],
                    machine_index = i,
                    start = inicio_minimo,
                    finish = fim,
                    duration = P[job_idx, i],
                ))
            end
        end
        makespan = conclusao[n, m]
    end

    return (
        model = model,
        instance = instancia.name,
        status = status,
        primal_status = primal,
        makespan = makespan,
        solve_time = tempo,
        gap = gap,
        sequence = [instancia.jobs[j] for j in sequencia_indices],
        sequence_indices = sequencia_indices,
        operations = operacoes,
        horizon = H,
        binary_variables = n * n,
    )
end

"""
    validar_solucao_flow_shop(instancia, solucao; atol = 1e-5)

Valida atribuicao dos jobs, precedencia entre maquinas, ausencia de sobreposicao
em cada maquina e consistencia do makespan.
"""
function validar_solucao_flow_shop(instancia, solucao; atol::Float64 = 1e-5)
    violacoes = String[]
    ops = solucao.operations
    n = instancia.n
    m = instancia.machines

    if length(solucao.sequence) != n
        push!(violacoes, "sequencia nao possui exatamente n jobs")
    end
    if length(unique(solucao.sequence)) != n
        push!(violacoes, "sequencia possui jobs repetidos ou ausentes")
    end
    for job in instancia.jobs
        if !(job in solucao.sequence)
            push!(violacoes, "job ausente na sequencia: $job")
        end
    end
    if length(ops) != n * m
        push!(violacoes, "numero de operacoes difere de n*m")
    end

    por_pos_maquina = Dict((op.position, op.machine_index) => op for op in ops)
    for k in 1:n
        if k > length(solucao.sequence)
            continue
        end
        job = solucao.sequence[k]
        job_idx = findfirst(==(job), instancia.jobs)
        if isnothing(job_idx)
            push!(violacoes, "job desconhecido na sequencia: $job")
            continue
        end
        for i in 1:m
            chave = (k, i)
            if !haskey(por_pos_maquina, chave)
                push!(violacoes, "operacao ausente na posicao $k maquina $i")
                continue
            end
            op = por_pos_maquina[chave]
            if op.job != job
                push!(violacoes, "job inconsistente na posicao $k maquina $i")
            end
            if abs((op.finish - op.start) - instancia.processing_times[job_idx, i]) > atol
                push!(violacoes, "duracao inconsistente para job $job na maquina $(instancia.machine_names[i])")
            end
            if op.start < -atol
                push!(violacoes, "inicio negativo para job $job na maquina $(instancia.machine_names[i])")
            end
        end
    end

    for k in 1:n, i in 2:m
        atual = get(por_pos_maquina, (k, i), nothing)
        anterior_maquina = get(por_pos_maquina, (k, i - 1), nothing)
        if atual !== nothing && anterior_maquina !== nothing && atual.start + atol < anterior_maquina.finish
            push!(violacoes, "precedencia entre maquinas violada na posicao $k: $(anterior_maquina.machine) antes de $(atual.machine)")
        end
    end

    for i in 1:m, k in 2:n
        atual = get(por_pos_maquina, (k, i), nothing)
        anterior_posicao = get(por_pos_maquina, (k - 1, i), nothing)
        if atual !== nothing && anterior_posicao !== nothing && atual.start + atol < anterior_posicao.finish
            push!(violacoes, "sobreposicao na maquina $(instancia.machine_names[i]) entre posicoes $(k - 1) e $k")
        end
    end

    makespan_calculado = isempty(ops) ? NaN : maximum(op.finish for op in ops)
    if !isnan(solucao.makespan) && abs(solucao.makespan - makespan_calculado) > atol
        push!(violacoes, "makespan reportado difere do makespan calculado")
    end

    return (
        valid = isempty(violacoes),
        violations = violacoes,
        makespan_calculado = makespan_calculado,
    )
end

function imprimir_resumo_flow_shop(solucao, validacao)
    println("Flow Shop Scheduling - ", solucao.instance)
    println("  status: ", solucao.status)
    println("  primal_status: ", solucao.primal_status)
    println("  sequencia: ", join(solucao.sequence, " -> "))
    println("  makespan: ", round(solucao.makespan; digits = 6))
    println("  tempo_execucao_s: ", round(solucao.solve_time; digits = 6))
    println("  gap: ", isnan(solucao.gap) ? "NA" : string(round(solucao.gap; digits = 6)))
    println("  binarios: ", solucao.binary_variables)
    println("  validacao: ", validacao.valid)
    if !validacao.valid
        println("  violacoes: ", validacao.violations)
    end
    println("  operacoes:")
    for op in sort(solucao.operations, by = o -> (o.position, o.machine_index))
        @printf("    pos=%2d job=%-4s maq=%-3s inicio=%8.3f fim=%8.3f dur=%3d\n",
            op.position, op.job, op.machine, op.start, op.finish, op.duration)
    end
end


## 7.3 Exemplo de solução

Nesta etapa, o modelo de Flow Shop Scheduling sera executado para a menor instancia disponivel, `problem_3m_10j.csv`. A instancia possui 10 jobs e 3 maquinas, e todos os jobs seguem a ordem comum definida pelas colunas do CSV: `M1 -> M2 -> M3`.

A tabela impressa pela celula mostra, para cada posicao da sequencia e maquina, o job alocado, inicio, termino e duracao. A validacao confere se a sequencia contem todos os jobs uma unica vez, se as precedencias entre maquinas foram respeitadas, se nao ha sobreposicao em cada maquina e se o makespan reportado corresponde ao maior tempo de termino encontrado.


In [ ]:
# Execucao inicial do Flow Shop Scheduling em uma instancia pequena

inst_flow_pequena = ler_instancia_fssp(joinpath(FSSP_DIR, "problem_3m_10j.csv"))
sol_flow_pequena = resolver_flow_shop(inst_flow_pequena; time_limit = 60.0, silent = true)
validacao_flow_pequena = validar_solucao_flow_shop(inst_flow_pequena, sol_flow_pequena)

imprimir_resumo_flow_shop(sol_flow_pequena, validacao_flow_pequena)


## Testes dos leitores

Esta secao executa apenas a leitura de uma instancia pequena de cada familia. Os testes nao modificam os arquivos de entrada e nao resolvem modelos de otimizacao.


In [ ]:
# Testes de leitura das instancias pequenas

machine_teste = ler_instancia_machine(joinpath(MACHINE_DIR, "inst_n05_s01.json"))
jsp_teste = ler_instancia_jsp(joinpath(JSP_DIR, "instances", "ft06"))
fssp_teste = ler_instancia_fssp(joinpath(FSSP_DIR, "problem_3m_10j.csv"))

println("Machine Scheduling")
println("  instancia: ", machine_teste.name)
println("  tarefas: ", machine_teste.n)
println("  maquinas: ", machine_teste.machines)
println("  operacoes: ", machine_teste.operations)

println("Job Shop Scheduling")
println("  instancia: ", jsp_teste.name)
println("  tarefas: ", jsp_teste.n)
println("  maquinas: ", jsp_teste.machines)
println("  operacoes: ", jsp_teste.operations)

println("Flow Shop Scheduling")
println("  instancia: ", fssp_teste.name)
println("  tarefas: ", fssp_teste.n)
println("  maquinas: ", fssp_teste.machines)
println("  operacoes: ", fssp_teste.operations)


# 8. Metodologia dos experimentos

A metodologia devera definir:

1. quais instancias serao executadas;
2. limites de tempo por instancia;
3. parametros do HiGHS;
4. criterios de comparacao;
5. metricas registradas, como status, valor objetivo, gap, tempo de solucao e numero de variaveis binarias;
6. forma de armazenamento e apresentacao dos resultados.

Nenhum resultado sera registrado sem execucao real do codigo.

## Execução em lote

A rotina de lote abaixo padroniza a execucao dos tres modelos. Cada linha de resultado registra problema, instancia, dimensoes, status do solver, valor objetivo ou makespan, melhor limite conhecido, gap relativo, tempo, tamanho do modelo, indicacao de limite de tempo e eventual mensagem de erro.

Os resultados sao salvos incrementalmente apos cada instancia. Se o CSV de saida ja contem uma instancia, ela e ignorada, permitindo continuar uma execucao interrompida sem repetir trabalho. Durante o lote, o log do HiGHS fica desativado e apenas uma linha resumida e impressa por instancia.


In [ ]:
# Interface padronizada e execucao em lote

const RESULT_COLUMNS = [
    "problema",
    "instancia",
    "tarefas",
    "maquinas",
    "operacoes",
    "status_terminacao",
    "status_primal",
    "valor_objetivo_ou_makespan",
    "melhor_limite",
    "gap_relativo",
    "tempo_solucao",
    "quantidade_variaveis",
    "quantidade_restricoes",
    "limite_tempo",
    "mensagem_erro",
]

const RESULTADOS_POR_PROBLEMA = Dict(
    "machine" => joinpath(@__DIR__, "resultados_machine_scheduling.csv"),
    "job" => joinpath(@__DIR__, "resultados_job_shop.csv"),
    "flow" => joinpath(@__DIR__, "resultados_flow_shop.csv"),
)
const RESULTADOS_COMPLETOS = joinpath(@__DIR__, "resultados_completos.csv")

function normalizar_problema(problema::AbstractString)
    p = lowercase(strip(problema))
    if p in ["machine", "machine scheduling", "machinescheduling"]
        return "machine"
    elseif p in ["job", "job shop", "job shop scheduling", "jsp"]
        return "job"
    elseif p in ["flow", "flow shop", "flow shop scheduling", "fssp"]
        return "flow"
    else
        error("Problema desconhecido: $problema")
    end
end

function nome_problema_padrao(chave::AbstractString)
    chave == "machine" && return "Machine Scheduling"
    chave == "job" && return "Job Shop Scheduling"
    chave == "flow" && return "Flow Shop Scheduling"
    error("Problema desconhecido: $chave")
end

function arquivos_instancias(problema::AbstractString, pasta::AbstractString)
    chave = normalizar_problema(problema)
    if !isdir(pasta)
        error("Pasta de instancias nao encontrada: $pasta")
    end

    raiz = pasta
    if chave == "job" && isdir(joinpath(pasta, "instances"))
        raiz = joinpath(pasta, "instances")
    end

    arquivos = String[]
    for path in sort(readdir(raiz; join = true))
        isfile(path) || continue
        if chave == "machine"
            endswith(lowercase(path), ".json") || continue
        elseif chave == "flow"
            endswith(lowercase(path), ".csv") || continue
        elseif chave == "job"
            basename(path) == "instances.json" && continue
        end
        push!(arquivos, path)
    end
    return arquivos
end

function csv_escape(valor)
    if valor === nothing
        texto = ""
    elseif valor isa AbstractFloat && isnan(valor)
        texto = ""
    else
        texto = string(valor)
    end
    if occursin(',', texto) || occursin('"', texto) || occursin('\n', texto)
        return "\"" * replace(texto, "\"" => "\"\"") * "\""
    end
    return texto
end

function escrever_cabecalho_se_necessario(path::AbstractString)
    if !isfile(path) || filesize(path) == 0
        open(path, "w") do io
            println(io, join(RESULT_COLUMNS, ","))
        end
    end
    return nothing
end

function append_resultado_csv!(path::AbstractString, resultado::NamedTuple)
    escrever_cabecalho_se_necessario(path)
    open(path, "a") do io
        println(io, join([csv_escape(getproperty(resultado, Symbol(col))) for col in RESULT_COLUMNS], ","))
    end
    return nothing
end

function chaves_resultados_existentes(path::AbstractString)
    chaves = Set{Tuple{String, String}}()
    isfile(path) || return chaves
    primeira = true
    for linha in eachline(path)
        if primeira
            primeira = false
            continue
        end
        isempty(strip(linha)) && continue
        partes = split(linha, ','; limit = 3)
        length(partes) >= 2 || continue
        push!(chaves, (partes[1], partes[2]))
    end
    return chaves
end

function contar_restricoes_modelo(model::Model)
    total = 0
    for (F, S) in list_of_constraint_types(model)
        total += num_constraints(model, F, S)
    end
    return total
end

function melhor_limite_modelo(model::Model)
    try
        return objective_bound(model)
    catch
        return NaN
    end
end

function resultado_erro(problema_nome, instancia_nome; mensagem, tempo = NaN)
    return (
        problema = problema_nome,
        instancia = instancia_nome,
        tarefas = "",
        maquinas = "",
        operacoes = "",
        status_terminacao = "ERRO",
        status_primal = "",
        valor_objetivo_ou_makespan = NaN,
        melhor_limite = NaN,
        gap_relativo = NaN,
        tempo_solucao = tempo,
        quantidade_variaveis = "",
        quantidade_restricoes = "",
        limite_tempo = false,
        mensagem_erro = mensagem,
    )
end

function executar_instancia_padronizada(problema::AbstractString, path::AbstractString;
        time_limit::Float64 = DEFAULT_TIME_LIMIT, gap_relativo = nothing)
    chave = normalizar_problema(problema)
    problema_nome = nome_problema_padrao(chave)
    tempo_total = @elapsed begin
        if chave == "machine"
            instancia = ler_instancia_machine(path)
            solucao = resolver_machine_scheduling(instancia; time_limit = time_limit,
                gap_relativo = gap_relativo, silent = true)
            valor = solucao.objective
        elseif chave == "job"
            instancia = ler_instancia_jsp(path)
            solucao = resolver_job_shop(instancia; time_limit = time_limit,
                gap_relativo = gap_relativo, silent = true)
            valor = solucao.makespan
        else
            instancia = ler_instancia_fssp(path)
            solucao = resolver_flow_shop(instancia; time_limit = time_limit,
                gap_relativo = gap_relativo, silent = true)
            valor = solucao.makespan
        end
    end

    status_txt = string(solucao.status)
    return (
        problema = problema_nome,
        instancia = instancia.name,
        tarefas = instancia.n,
        maquinas = instancia.machines,
        operacoes = instancia.operations,
        status_terminacao = status_txt,
        status_primal = string(solucao.primal_status),
        valor_objetivo_ou_makespan = valor,
        melhor_limite = melhor_limite_modelo(solucao.model),
        gap_relativo = solucao.gap,
        tempo_solucao = solucao.solve_time,
        quantidade_variaveis = num_variables(solucao.model),
        quantidade_restricoes = contar_restricoes_modelo(solucao.model),
        limite_tempo = status_txt == "TIME_LIMIT",
        mensagem_erro = "",
    )
end

function imprimir_linha_resumida(resultado)
    valor = resultado.valor_objetivo_ou_makespan
    valor_txt = valor isa AbstractFloat && isnan(valor) ? "NA" : string(round(float(valor); digits = 6))
    gap = resultado.gap_relativo
    gap_txt = gap isa AbstractFloat && isnan(gap) ? "NA" : string(round(float(gap); digits = 6))
    tempo = resultado.tempo_solucao
    tempo_txt = tempo isa AbstractFloat && isnan(tempo) ? "NA" : string(round(float(tempo); digits = 3))
    println(resultado.problema, " | ", resultado.instancia,
        " | status=", resultado.status_terminacao,
        " | valor=", valor_txt,
        " | gap=", gap_txt,
        " | tempo=", tempo_txt, "s",
        isempty(resultado.mensagem_erro) ? "" : " | erro=" * resultado.mensagem_erro)
end

"""
    executar_lote(problema, pasta; output_csv = nothing, completos_csv = RESULTADOS_COMPLETOS,
                  time_limit = DEFAULT_TIME_LIMIT, gap_relativo = nothing, limite = nothing,
                  somente = nothing)

Executa as instancias compativeis de `pasta`, salva cada resultado imediatamente
e pula instancias ja presentes no CSV especifico do problema. O argumento opcional
`somente` limita a execucao a uma lista de nomes de arquivos, usado apenas para
testes pequenos e controlados.
"""
function executar_lote(problema::AbstractString, pasta::AbstractString;
        output_csv = nothing, completos_csv::AbstractString = RESULTADOS_COMPLETOS,
        time_limit::Float64 = DEFAULT_TIME_LIMIT, gap_relativo = nothing, limite = nothing,
        somente = nothing)
    chave = normalizar_problema(problema)
    problema_nome = nome_problema_padrao(chave)
    output = output_csv === nothing ? RESULTADOS_POR_PROBLEMA[chave] : output_csv
    arquivos = arquivos_instancias(chave, pasta)
    if somente !== nothing
        permitidos = Set(String.(somente))
        arquivos = [path for path in arquivos if basename(path) in permitidos || splitext(basename(path))[1] in permitidos]
    end
    if limite !== nothing
        arquivos = arquivos[1:min(Int(limite), length(arquivos))]
    end

    ja_executadas = chaves_resultados_existentes(output)
    resultados = NamedTuple[]

    for path in arquivos
        instancia_nome = chave == "job" ? basename(path) : splitext(basename(path))[1]
        chave_csv = (problema_nome, instancia_nome)
        if chave_csv in ja_executadas
            println(problema_nome, " | ", instancia_nome, " | ja registrado, ignorado")
            continue
        end

        resultado = try
            executar_instancia_padronizada(chave, path; time_limit = time_limit, gap_relativo = gap_relativo)
        catch err
            resultado_erro(problema_nome, instancia_nome; mensagem = sprint(showerror, err))
        end

        append_resultado_csv!(output, resultado)
        append_resultado_csv!(completos_csv, resultado)
        push!(resultados, resultado)
        push!(ja_executadas, (resultado.problema, resultado.instancia))
        imprimir_linha_resumida(resultado)
    end

    return resultados
end


## Teste limitado da rotina de lote

A celula abaixo documenta como executar um teste limitado com duas instancias por problema. Ela permanece comentada para evitar sobrescrever ou modificar os CSVs finais durante uma execucao integral do notebook.


In [ ]:
# Teste limitado opcional: descomente para executar duas instancias por problema.

# resultados_teste_machine = executar_lote("machine", MACHINE_DIR;
#     time_limit = 15.0, gap_relativo = 0.01, somente = ["inst_book.json", "inst_n05_s01.json"])
# resultados_teste_job = executar_lote("job", JSP_DIR;
#     time_limit = 15.0, gap_relativo = 0.01, somente = ["ft06", "la01"])
# resultados_teste_flow = executar_lote("flow", FSSP_DIR;
#     time_limit = 15.0, gap_relativo = 0.01, somente = ["problem_3m_10j.csv", "problem_3m_20j.csv"])


# 9. Resultados completos

Esta secao usa exclusivamente os dados ja registrados em `desenvolvimento/resultados_completos.csv`. Nenhum valor e preenchido manualmente: as estatisticas abaixo sao calculadas a partir das colunas do CSV consolidado.


In [ ]:
# Resumo estatistico a partir de resultados_completos.csv

function ler_csv_simples(path::AbstractString)
    validar_arquivo_legivel(path)
    linhas = collect(eachline(path))
    isempty(linhas) && error("CSV vazio: $path")
    header = split(linhas[1], ",")
    rows = Vector{Dict{String, String}}()
    for linha in linhas[2:end]
        isempty(strip(linha)) && continue
        valores = parse_linha_csv(linha)
        if length(valores) != length(header)
            error("Linha CSV com numero de colunas inesperado em $path: $linha")
        end
        push!(rows, Dict(header[i] => valores[i] for i in eachindex(header)))
    end
    return rows
end

function parse_linha_csv(linha::AbstractString)
    valores = String[]
    atual = IOBuffer()
    em_aspas = false
    i = firstindex(linha)
    while i <= lastindex(linha)
        c = linha[i]
        if c == '"'
            prox = nextind(linha, i)
            if em_aspas && prox <= lastindex(linha) && linha[prox] == '"'
                print(atual, '"')
                i = prox
            else
                em_aspas = !em_aspas
            end
        elseif c == ',' && !em_aspas
            push!(valores, String(take!(atual)))
        else
            print(atual, c)
        end
        i = nextind(linha, i)
    end
    push!(valores, String(take!(atual)))
    return valores
end

function numero_ou_nan(row, coluna)
    texto = strip(get(row, coluna, ""))
    isempty(texto) && return NaN
    try
        return parse(Float64, texto)
    catch
        return NaN
    end
end

function media(vals)
    limpos = [v for v in vals if !isnan(v)]
    isempty(limpos) ? NaN : sum(limpos) / length(limpos)
end

function mediana(vals)
    limpos = sort([v for v in vals if !isnan(v)])
    n = length(limpos)
    if n == 0
        return NaN
    elseif isodd(n)
        return limpos[(n + 1) ÷ 2]
    else
        return (limpos[n ÷ 2] + limpos[n ÷ 2 + 1]) / 2
    end
end

function menor(vals)
    limpos = [v for v in vals if !isnan(v)]
    isempty(limpos) ? NaN : minimum(limpos)
end

function maior(vals)
    limpos = [v for v in vals if !isnan(v)]
    isempty(limpos) ? NaN : maximum(limpos)
end

function fmt(v; digits = 3)
    isnan(v) ? "NA" : string(round(v; digits = digits))
end

function resumo_por_problema(rows)
    problemas = sort(unique(row["problema"] for row in rows))
    resumo = NamedTuple[]
    for problema in problemas
        grupo = [row for row in rows if row["problema"] == problema]
        tempos = [numero_ou_nan(row, "tempo_solucao") for row in grupo]
        gaps_nao_otimos = [numero_ou_nan(row, "gap_relativo") for row in grupo
            if row["status_terminacao"] != "OPTIMAL" && !isnan(numero_ou_nan(row, "gap_relativo"))]
        push!(resumo, (
            problema = problema,
            total_instancias = length(grupo),
            solucoes_otimas = count(row -> row["status_terminacao"] == "OPTIMAL", grupo),
            solucoes_viaveis_nao_otimas = count(row -> row["status_primal"] == "FEASIBLE_POINT" && row["status_terminacao"] != "OPTIMAL", grupo),
            limites_tempo = count(row -> lowercase(row["limite_tempo"]) == "true" || row["status_terminacao"] == "TIME_LIMIT", grupo),
            erros = count(row -> !isempty(strip(row["mensagem_erro"])) || row["status_terminacao"] == "ERRO", grupo),
            tempo_medio = media(tempos),
            tempo_mediano = mediana(tempos),
            menor_tempo = menor(tempos),
            maior_tempo = maior(tempos),
            gap_medio_nao_otimas = media(gaps_nao_otimos),
            media_tarefas = media([numero_ou_nan(row, "tarefas") for row in grupo]),
            media_maquinas = media([numero_ou_nan(row, "maquinas") for row in grupo]),
            media_variaveis = media([numero_ou_nan(row, "quantidade_variaveis") for row in grupo]),
            media_restricoes = media([numero_ou_nan(row, "quantidade_restricoes") for row in grupo]),
        ))
    end
    return resumo
end

function imprimir_resumo_experimentos(resumo)
    cabecalho = [
        "Problema", "Inst.", "Otimas", "Viaveis nao otimas", "Time limit", "Erros",
        "Tempo medio", "Tempo mediano", "Menor tempo", "Maior tempo", "Gap medio nao otimas",
        "Media tarefas", "Media maquinas", "Media variaveis", "Media restricoes",
    ]
    println(join(cabecalho, " | "))
    println(join(fill("---", length(cabecalho)), " | "))
    for r in resumo
        println(join([
            r.problema,
            string(r.total_instancias),
            string(r.solucoes_otimas),
            string(r.solucoes_viaveis_nao_otimas),
            string(r.limites_tempo),
            string(r.erros),
            fmt(r.tempo_medio),
            fmt(r.tempo_mediano),
            fmt(r.menor_tempo),
            fmt(r.maior_tempo),
            fmt(r.gap_medio_nao_otimas; digits = 4),
            fmt(r.media_tarefas),
            fmt(r.media_maquinas),
            fmt(r.media_variaveis),
            fmt(r.media_restricoes),
        ], " | "))
    end
end

linhas_resultados = ler_csv_simples(RESULTADOS_COMPLETOS)
resumo_experimentos = resumo_por_problema(linhas_resultados)
imprimir_resumo_experimentos(resumo_experimentos)


# 10. Comparação dos problemas

Os graficos abaixo usam somente `resultados_completos.csv`. Como os tempos e gaps variam bastante entre os problemas, os graficos por instancia sao separados por problema para manter a leitura das escalas.


In [ ]:
# Graficos para analise dos resultados completos

using Plots

if !@isdefined(linhas_resultados)
    linhas_resultados = ler_csv_simples(RESULTADOS_COMPLETOS)
end

function grupos_por_problema(rows)
    problemas = sort(unique(row["problema"] for row in rows))
    return [(problema, [row for row in rows if row["problema"] == problema]) for problema in problemas]
end

function ordenar_por_instancia(rows)
    sort(rows, by = row -> row["instancia"])
end

function serie_numerica(rows, coluna)
    [numero_ou_nan(row, coluna) for row in rows]
end

function rotulos_instancias(rows)
    [row["instancia"] for row in rows]
end

grupos_resultados = grupos_por_problema(linhas_resultados)

# 1. Tempo de solucao por instancia, separado por problema.
graficos_tempo_instancia = Plots.Plot[]
for (problema, rows) in grupos_resultados
    ordenadas = ordenar_por_instancia(rows)
    push!(graficos_tempo_instancia, bar(
        rotulos_instancias(ordenadas),
        serie_numerica(ordenadas, "tempo_solucao");
        title = "Tempo por instancia - $problema",
        xlabel = "Instancia",
        ylabel = "Tempo de solucao (s)",
        legend = false,
        xrotation = 60,
        size = (900, 350),
    ))
end
plot(graficos_tempo_instancia...; layout = (length(graficos_tempo_instancia), 1), size = (950, 950))



In [ ]:
# 2. Tarefas versus tempo de solucao, com uma serie para cada problema.

p_tarefas_tempo = plot(
    title = "Tarefas versus tempo de solucao",
    xlabel = "Numero de tarefas",
    ylabel = "Tempo de solucao (s)",
    legend = :topleft,
)
for (problema, rows) in grupos_resultados
    scatter!(
        p_tarefas_tempo,
        serie_numerica(rows, "tarefas"),
        serie_numerica(rows, "tempo_solucao");
        label = problema,
        markersize = 5,
    )
end
p_tarefas_tempo



In [ ]:
# 3. Gap relativo por instancia, separado por problema.

graficos_gap_instancia = Plots.Plot[]
for (problema, rows) in grupos_resultados
    ordenadas = ordenar_por_instancia(rows)
    push!(graficos_gap_instancia, bar(
        rotulos_instancias(ordenadas),
        serie_numerica(ordenadas, "gap_relativo");
        title = "Gap relativo por instancia - $problema",
        xlabel = "Instancia",
        ylabel = "Gap relativo",
        legend = false,
        xrotation = 60,
        size = (900, 350),
    ))
end
plot(graficos_gap_instancia...; layout = (length(graficos_gap_instancia), 1), size = (950, 950))



In [ ]:
# 4. Quantidade de instancias por status, agrupada por problema.

status_unicos = sort(unique(row["status_terminacao"] for row in linhas_resultados))
problemas_unicos = [p for (p, _) in grupos_resultados]
matriz_status = zeros(Int, length(problemas_unicos), length(status_unicos))
for (i, problema) in enumerate(problemas_unicos)
    rows = [row for row in linhas_resultados if row["problema"] == problema]
    for (j, status) in enumerate(status_unicos)
        matriz_status[i, j] = count(row -> row["status_terminacao"] == status, rows)
    end
end

bar(
    problemas_unicos,
    matriz_status;
    label = reshape(status_unicos, 1, :),
    title = "Quantidade de instancias por status",
    xlabel = "Problema",
    ylabel = "Quantidade de instancias",
    legend = :topright,
    xrotation = 20,
    size = (900, 450),
)



In [ ]:
# 5. Comparacao do tempo medio entre os tres problemas.

tempos_medios = [media(serie_numerica(rows, "tempo_solucao")) for (_, rows) in grupos_resultados]
bar(
    problemas_unicos,
    tempos_medios;
    title = "Tempo medio de solucao por problema",
    xlabel = "Problema",
    ylabel = "Tempo medio de solucao (s)",
    legend = false,
    xrotation = 20,
    size = (850, 420),
)



## Tabela comparativa

A tabela comparativa e gerada a partir de `resultados_completos.csv`, usando os mesmos dados agregados da secao de resumo dos experimentos. Ela resume, por problema, o total de instancias, a quantidade de solucoes otimas, os limites de tempo, o tempo medio e o gap medio das solucoes nao otimas.


In [ ]:
# Tabela comparativa gerada a partir do CSV consolidado

if !@isdefined(resumo_experimentos)
    linhas_resultados = ler_csv_simples(RESULTADOS_COMPLETOS)
    resumo_experimentos = resumo_por_problema(linhas_resultados)
end

println("Problema | Instancias | Otimas | Viaveis nao otimas | Time limit | Tempo medio (s) | Gap medio nao otimas")
println("--- | ---: | ---: | ---: | ---: | ---: | ---:")
for r in resumo_experimentos
    println(join([
        r.problema,
        string(r.total_instancias),
        string(r.solucoes_otimas),
        string(r.solucoes_viaveis_nao_otimas),
        string(r.limites_tempo),
        fmt(r.tempo_medio),
        fmt(r.gap_medio_nao_otimas; digits = 4),
    ], " | "))
end


# 11. Discussão

Esta discussao utiliza exclusivamente os registros de `resultados_completos.csv`. Portanto, as conclusoes abaixo se referem ao conjunto de instancias efetivamente executado e aos parametros computacionais registrados: limite de tempo de 300 segundos por instancia e tolerancia relativa de gap de 1%.

## Resultados observados

O desempenho do HiGHS variou de forma clara entre os tres problemas. No Flow Shop Scheduling foram registradas 7 instancias, das quais 6 terminaram com status `OPTIMAL` e 1 atingiu `TIME_LIMIT`. Todas apresentaram ponto primal viavel. O tempo medio foi de 44,178 s, mas a mediana foi de apenas 0,619 s, indicando que a media foi fortemente influenciada pela instancia `problem_10m_20j`, que consumiu 300,249 s e terminou com gap relativo de 0,0241. Entre os tres problemas, Flow Shop foi o conjunto com menor tempo medio e menor gap medio entre solucoes nao otimas.

No Job Shop Scheduling foram registradas 10 instancias. Seis terminaram com status `OPTIMAL` e quatro atingiram o limite de tempo: `abz5`, `ft10`, `ft20` e `orb01`. Todas as instancias tiveram ponto primal viavel. O tempo medio foi de 190,464 s e a mediana foi de 235,695 s, mostrando que uma parcela relevante das instancias demandou tempos elevados. O gap medio das solucoes nao otimas foi 0,3118, com destaque para `ft20`, que terminou com gap 0,6008.

No Machine Scheduling foram registradas 31 instancias. Dezessete terminaram com status `OPTIMAL` e quatorze atingiram `TIME_LIMIT`; todas tiveram ponto primal viavel. O tempo medio foi de 150,670 s e a mediana foi de 119,012 s. As instancias nao otimas apresentaram gap medio de 0,8432, substancialmente maior do que os gaps medios nao otimos de Flow Shop e Job Shop. As instancias interrompidas por limite de tempo concentram-se nos tamanhos maiores, a partir das instancias com 15 tarefas.

A comparacao direta entre os tres problemas mostra que o Flow Shop foi o mais favoravel neste conjunto de dados, com 6 de 7 instancias otimas e apenas uma interrupcao por tempo. O Job Shop teve 6 de 10 instancias otimas, mas tempos medianos elevados e quatro interrupcoes. O Machine Scheduling teve a maior quantidade absoluta de instancias otimas, 17, mas tambem a maior quantidade de interrupcoes, 14, e os maiores gaps medios entre solucoes nao otimas.

## Possiveis explicacoes

O numero de tarefas parece estar associado ao aumento da dificuldade, especialmente em Machine Scheduling. Nesse problema, as instancias variam de 5 a 24 tarefas, com uma unica maquina. Apesar de haver apenas uma maquina, o modelo usa variaveis binarias associadas a pares de tarefas, o que faz o tamanho do modelo crescer rapidamente com o numero de tarefas. Isso aparece nos registros: a media de variaveis foi 148,548 e a media de restricoes foi 445,645, com maximos de 348 variaveis e 1044 restricoes. As instancias de maior porte foram justamente aquelas que mais frequentemente atingiram 300 segundos.

No Job Shop, o numero medio de tarefas foi 10,6 e o numero medio de maquinas foi 7,1. O tamanho medio dos modelos foi 448,6 variaveis e 1344,8 restricoes, com maximos de 1051 variaveis e 3152 restricoes. A presenca simultanea de precedencias tecnologicas e conflitos de capacidade em multiplas maquinas ajuda a explicar, como hipotese compatível com os dados, os tempos elevados e os limites de tempo observados. Entretanto, os dados tambem mostram que tamanho do modelo nao explica tudo isoladamente: algumas instancias com dimensoes parecidas tiveram comportamentos diferentes.

No Flow Shop, apesar de haver instancias com ate 30 tarefas e 10 maquinas, a formulacao posicional usada registrou desempenho melhor neste conjunto. A media de variaveis foi 439,571 e a media de restricoes foi 741,286. A estrutura de sequencia comum entre maquinas pode ter contribuido para uma formulacao mais regular do que a formulacao disjuntiva do Job Shop, mas esta interpretacao deve ser tomada como uma explicacao possivel, nao como conclusao geral sobre todos os casos de Flow Shop.

A relacao entre quantidade de variaveis, restricoes e tempo e observavel, mas nao perfeitamente deterministica. Em geral, problemas com modelos maiores tenderam a exigir mais tempo, especialmente Job Shop e as maiores instancias de Machine Scheduling. Ainda assim, os tempos tambem dependem da estrutura combinatoria de cada instancia, da qualidade dos limites obtidos pelo solver e da capacidade de fechar o gap dentro da tolerancia configurada.

## Comportamento do gap e efeito da configuracao

A configuracao de gap relativo maximo de 1% afeta diretamente a interpretacao do status `OPTIMAL`. Nos registros, ha instancias com status `OPTIMAL` e gap positivo abaixo de 1%, o que significa otimalidade dentro da tolerancia configurada, nao necessariamente prova de gap numericamente zero. Isso aparece, por exemplo, em varias instancias de Machine Scheduling, Job Shop e Flow Shop com gap entre 0 e 0,01.

Para instancias interrompidas por limite de tempo, os gaps indicam a distancia relativa remanescente entre a melhor solucao viavel e o melhor limite conhecido pelo solver. O Flow Shop teve apenas uma instancia nao otima, com gap 0,0241. O Job Shop teve quatro instancias nao otimas, com gap medio 0,3118. O Machine Scheduling teve quatorze instancias nao otimas, com gap medio 0,8432. Assim, neste experimento, o limite de 300 segundos foi suficiente para obter solucoes viaveis em todos os casos, mas nao foi suficiente para fechar o gap em uma parcela relevante das instancias de Machine Scheduling e Job Shop.

## Limitacoes do experimento

Os resultados nao permitem afirmar que um problema e sempre mais facil que outro em termos gerais. A comparacao vale para estas instancias, estas formulacoes e esta configuracao do HiGHS. O conjunto de Flow Shop, por exemplo, tem apenas 7 instancias, enquanto Machine Scheduling tem 31; portanto, as medias agregadas devem ser comparadas com cautela.

Outra limitacao e que foram usadas formulacoes MILP diretas. Em Machine Scheduling e Job Shop, as restricoes disjuntivas introduzem variaveis binarias por pares de tarefas ou operacoes conflitantes, o que pode produzir modelos grandes e relaxacoes lineares fracas. No Flow Shop, a formulacao posicional explora a estrutura de permutacao comum, o que pode favorecer o desempenho nesse conjunto, mas nao elimina a natureza combinatoria do problema.

Por fim, os CSVs registram metricas agregadas de execucao, mas nao registram a programacao completa de cada solucao. Assim, a analise computacional aqui discute desempenho, status, gaps e dimensoes dos modelos, mas nao examina visualmente a estrutura das agendas finais de todas as instancias. Essa restricao deve ser considerada ao interpretar a qualidade operacional das solucoes alem dos indicadores fornecidos pelo solver.


# 12. Conclusão

A atividade desenvolveu e analisou computacionalmente tres problemas classicos de escalonamento: Machine Scheduling, Job Shop Scheduling e Flow Shop Scheduling. As implementacoes foram realizadas em Julia, com modelagem em JuMP e resolucao pelo solver HiGHS, mantendo uma interface padronizada para execucao em lote, registro de status, tempos, gaps, limites conhecidos e dimensoes dos modelos.

Ao todo, foram analisadas 48 instancias registradas em `resultados_completos.csv`: 31 de Machine Scheduling, 10 de Job Shop Scheduling e 7 de Flow Shop Scheduling. Em todos os casos, o solver encontrou ponto primal viavel. O Flow Shop apresentou o comportamento computacional mais favoravel neste conjunto, com 6 de 7 instancias resolvidas dentro da tolerancia configurada. O Job Shop teve 6 de 10 instancias com status `OPTIMAL`, enquanto o Machine Scheduling teve 17 de 31. Apesar de Machine Scheduling ter o maior numero absoluto de instancias otimas, tambem concentrou a maior quantidade de interrupcoes por limite de tempo e os maiores gaps medios entre as solucoes nao otimas.

Considerando os dados obtidos, o problema que apresentou maior dificuldade foi o Machine Scheduling nas instancias maiores, especialmente a partir dos casos com 15 tarefas. Essa conclusao e restrita ao conjunto de instancias, formulacoes e parametros usados no experimento. O Job Shop tambem apresentou dificuldade relevante, com tempos elevados e quatro instancias interrompidas por limite de tempo. O Flow Shop, por sua vez, foi comparativamente mais estavel neste conjunto de testes.

As principais limitacoes do estudo decorrem do uso de formulacoes MILP diretas, com variaveis binarias associadas a decisoes combinatorias de sequenciamento, e da configuracao de tempo maximo de 300 segundos por instancia. Essa configuracao permitiu obter solucoes viaveis para todas as instancias, mas nao foi suficiente para fechar o gap em todos os casos. Outra limitacao e que os arquivos de resultados armazenam metricas agregadas, mas nao registram a programacao completa das solucoes, o que restringe validacoes posteriores sem nova execucao dos modelos.

Como trabalhos futuros, seria natural comparar as formulacoes usadas com modelos alternativos mais fortes, testar diferentes valores de limite de tempo e tolerancia de gap, incluir heuristicas construtivas ou metaheuristicas para fornecer solucoes iniciais, registrar as programacoes completas em arquivos auxiliares e ampliar a analise com visualizacoes de Gantt para instancias representativas. Tambem seria interessante avaliar separadamente o impacto do numero de tarefas, maquinas e restricoes disjuntivas sobre o desempenho do solver.
